In [2]:
import sys
import os
ROOT_DIR = "/work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG"
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

os.environ["PYTHONWARNINGS"] = "ignore"  # 全局忽略所有 warnings

print(ROOT_DIR)

/work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG


In [1]:
import sys
# 本地 CACE-SOG 源码目录（必须先于 import cace，否则会 ModuleNotFoundError）
_ROOT = "/work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG"
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

from ase.io import read, write
import torch
import cace

device = "cpu"  # 或 "cuda"

# 1. 自己用 weights_only=False 先把模型加载出来
model = torch.load(
    '/work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG/fit-4hdnnp-NaCl/loss_data/Nacl_model.pth',
    map_location=device,
    weights_only=False,  # 关键：显式关闭安全模式（前提是你信任这个 checkpoint）
)

# 2. 把 nn.Module 直接传给 EvaluateTask（它支持这一用法）
evaluator = cace.tasks.EvaluateTask(
    model_path=model,
    device=device,
    energy_key='CACE_energy',
    forces_key='CACE_forces',
)

data = read('/work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG/fit-4hdnnp-NaCl/NaCl.xyz',':')

pre = evaluator(data)
pre['energy']

KeyboardInterrupt: 

In [11]:
import numpy as np
def get_property(atoms, info_name, atomic_energies=None):
    if info_name == 'energy' and atomic_energies is not None:
        # 对于每个结构，计算总能量并扣除基态能量
        ene_results = []
        num_results = []
        for a in atoms:
            energy = a.info.get(info_name, None)
            if energy is None:
                raise ValueError(f"Property '{info_name}' not found in atoms info.")
            # 获取原子序数
            atomic_numbers = a.get_atomic_numbers()
            # 扣除基态能量
            energy -= sum(atomic_energies.get(Z, 0) for Z in atomic_numbers)
            ene_results.append(energy)
            num_results.append(len(atomic_numbers))
        return np.array(ene_results),np.array(num_results)

In [12]:
atomic_energies={11: -4417.07609365649, 17: -12516.880649933015}
ref_energy,atom_num= get_property(data, 'energy', atomic_energies=atomic_energies)

NameError: name 'data' is not defined

In [30]:
pre['energy']

array([ 0.15400946,  0.0252136 ,  0.07934177, ...,  0.07319129,
       -0.02893162, -0.16877043], dtype=float32)

In [31]:
ref_energy

array([ 0.15517016,  0.02694292,  0.07818196, ...,  0.07356963,
       -0.02910438, -0.16642664])

In [32]:
np.sqrt(np.mean(((ref_energy-pre['energy'])/atom_num)**2))

0.0001433191843173955

In [33]:
def get_forces(atoms_list):
    forces_list = []
    for a in atoms_list:
        # 检查是否有力
        if 'forces' in a.arrays:
            forces_list.append(a.arrays['forces'])
        else:
            # 如果没有，可以用None或者空数组，但这里我们选择用None表示缺失
            forces_list.append(None)
    return forces_list

In [34]:
ref_forces = get_forces(data)

In [35]:
ref_force = np.concatenate(ref_forces, axis=0)

In [36]:
np.sqrt(np.mean((ref_force-pre['forces'])**2))

0.004016742096638881